# iDECOP-ICS — AI first pass for the plausibility rubric (A20, D26)

Scores Filter 1's four criteria (`formal-problem-definition.md` §4) for every asset except the Attacker node, and writes two files that Screen 3 imports:
- `ai_suggestions.csv` — one suggestion per asset;
- `ai_raw_outputs.jsonl` — the model's raw text, kept for audit.

The prompt, the rules and the scoring loop live in `src/plausibility/prompt.py`. This notebook only loads the model and calls them.

**Before you run it:** confirm the three blind cards in Screen 3 first (DMZ Jump Host, Historian, PLC-03 (RTU)). This notebook never prints an answer, but the CSV contains them.

**Run:**
1. *Runtime → Change runtime type → T4 GPU* (any GPU works).
2. *Runtime → Run all.* Most of the time goes on downloading the model (about 30 GB). After that, each asset takes about a minute. These are estimates; the first run will show the real figures.
3. When the last cell finishes, both files download.

**Then, on your machine:**
```
python scripts/import_plausibility_scores.py ai_suggestions.csv
```
Move both files into `data/review/` and commit them with the review.

**If the model will not load** (an out-of-memory error): set `MODEL_ID = "Qwen/Qwen3-8B"` in the next cell, then *Runtime → Restart session* and *Run all*. The restart matters: a failed load keeps its GPU memory, and Python would keep the previously imported prompt module. That fallback is pre-declared in D26, and the CSV records which model produced the suggestions. If no GPU is available within two weekends of finishing the blind cards, score the remaining cards directly in Screen 3 (route B, D26).

In [ ]:
MODEL_ID = "Qwen/Qwen3-14B"   # D12. Pre-declared fallback (D26): "Qwen/Qwen3-8B"
REPO = "https://github.com/nittsOG/idecop-ics.git"
MAX_NEW_TOKENS = 900           # one JSON object per asset; about 300 tokens are typical

In [ ]:
# Qwen3 needs transformers 4.51 or later. The upper bound pins the 4.x API this notebook was written
# against (argument names such as torch_dtype). Install before anything imports transformers.
!pip install -q "transformers>=4.51,<5" accelerate bitsandbytes

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again."
GPU = torch.cuda.get_device_name(0)
print(GPU, round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GiB")

In [ ]:
import subprocess, sys
!rm -rf idecop-ics && git clone -q {REPO} idecop-ics
COMMIT = subprocess.check_output(["git", "-C", "idecop-ics", "rev-parse", "HEAD"], text=True).strip()
sys.path.insert(0, "idecop-ics")
print("repository commit", COMMIT)

In [ ]:
import sqlite3
from src.plausibility import prompt as pp

conn = sqlite3.connect(":memory:")
for f in ("data/schema.sql", "data/seed.sql"):
    conn.executescript(open(f"idecop-ics/{f}").read())
contexts = [pp.asset_context(conn, i) for i in pp.assets_to_score(conn)]
print(len(contexts), "assets:", ", ".join(c["name"] for c in contexts))

In [ ]:
from huggingface_hub import model_info
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import transformers, accelerate, bitsandbytes

REVISION = model_info(MODEL_ID).sha          # pin the exact weights used
tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=REVISION, device_map="auto", torch_dtype=torch.float16,
    # 14.8B parameters are about 29.6 GB at 16-bit, too large for a 16 GB T4, so the
    # weights load in 4-bit. The embeddings and output layer stay at 16-bit, so the
    # total is about 10 GB. The T4 lists no BF16 support, so everything not quantised,
    # and all compute, is FP16 (sources.md 364-366 and 376).
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16))
model.eval()
MODEL_TAG = f"{MODEL_ID}@{REVISION[:12]} (4-bit nf4, greedy)"
print("loaded", MODEL_TAG, "| GPU memory in use:", round(torch.cuda.memory_allocated() / 2**30, 1), "GiB")

In [ ]:
def generate(messages):
    # Non-thinking mode with greedy decoding, so a rerun on the same setup should give the same text
    # (GPU kernels are not guaranteed to be deterministic, which is one reason the raw outputs are kept).
    # The model card recommends sampling (temperature 0.7) for non-thinking mode and warns against
    # greedy decoding in thinking mode, which is off here (sources.md 364). An output that loops or
    # breaks the JSON fails to parse, gets one retry, and otherwise leaves the card to the reviewer.
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

from datetime import datetime, timezone
GENERATED_AT = datetime.now(timezone.utc).isoformat(timespec="seconds")
rows, raw = pp.run(contexts, generate, MODEL_TAG, COMMIT, GENERATED_AT)   # prints progress, never answers

In [ ]:
import csv, json
from google.colab import files

with open("ai_suggestions.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=pp.CSV_FIELDS)
    w.writeheader()
    w.writerows(rows)
meta = {"model": MODEL_ID, "revision": REVISION, "repo_commit": COMMIT, "generated_at": GENERATED_AT,
        "gpu": GPU, "transformers": transformers.__version__, "torch": torch.__version__,
        "accelerate": accelerate.__version__, "bitsandbytes": bitsandbytes.__version__,
        "decoding": "greedy, enable_thinking=False", "max_new_tokens": MAX_NEW_TOKENS,
        "system_prompt": pp.SYSTEM_PROMPT}
with open("ai_raw_outputs.jsonl", "w", encoding="utf-8") as f:
    f.write(json.dumps({"meta": meta}) + "\n")
    for r in raw:
        f.write(json.dumps(r) + "\n")

failed = [r["asset_name"] for r in rows if r["parse_status"] != "ok"]
print(f"{len(rows) - len(failed)} of {len(rows)} assets have a suggestion.")
if failed:
    print("No usable answer for:", ", ".join(failed), "- score those directly in Screen 3, with a reason.")
files.download("ai_suggestions.csv")
files.download("ai_raw_outputs.jsonl")